# LLM Eval: Movie Remakes Structural Similarity

This notebook:
1. Reads OpenAI API key from `openai_key.txt`
2. Loads eval pairs from `data/eval_data/eval_story_pairs_200.csv`
3. Filters `movie_remakes` pairs
4. Calls `gpt-5` to rate structural similarity from **1 to 10**
5. Saves results to `data/eval_results/movie_remakes_llm_structural_scores.json`


In [1]:
from pathlib import Path
import json
import time
import re
from typing import Any

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

In [6]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
EVAL_INPUT_CSV = Path("/Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv")
RESULTS_DIR = Path("/Users/shayan/Projects/NarrativeSimilarity/src/eval_results")
RESULTS_PATH = RESULTS_DIR / 'movie_remakes_llm_structural_scores.json'
KEY_PATH = PROJECT_ROOT / 'openai_key.txt'

MODEL_NAME = 'gpt-5'

REQUEST_SLEEP_SECONDS = 0.25
MAX_RETRIES = 4

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('EVAL_INPUT_CSV:', EVAL_INPUT_CSV)
print('RESULTS_PATH  :', RESULTS_PATH)
print('KEY_PATH      :', KEY_PATH)


EVAL_INPUT_CSV: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv
RESULTS_PATH  : /Users/shayan/Projects/NarrativeSimilarity/src/eval_results/movie_remakes_llm_structural_scores.json
KEY_PATH      : /Users/shayan/Projects/NarrativeSimilarity/src/openai_key.txt


In [7]:
if not KEY_PATH.exists():
    KEY_PATH = PROJECT_ROOT.parent / 'openai_key.txt'

if not KEY_PATH.exists():
    raise FileNotFoundError(f'OpenAI key file not found: {KEY_PATH}')

api_key = KEY_PATH.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'OpenAI key file is empty: {KEY_PATH}')

client = OpenAI(api_key=api_key)

print(f'OpenAI client initialized. Key file: {KEY_PATH}')


OpenAI client initialized. Key file: /Users/shayan/Projects/NarrativeSimilarity/openai_key.txt


In [8]:
if not EVAL_INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {EVAL_INPUT_CSV}')

eval_df = pd.read_csv(EVAL_INPUT_CSV)
movie_df = eval_df[eval_df['dataset'] == 'movie_remakes'].copy().reset_index(drop=True)

print('Total rows in eval file:', len(eval_df))
print('Movie remakes rows:', len(movie_df))
movie_df[['pair_id', 'pair_type', 'label']].head()

Total rows in eval file: 200
Movie remakes rows: 100


,pair_id,pair_type,label
0,movie_remakes__0100,random_negative,0
1,movie_remakes__0101,positive,1
2,movie_remakes__0102,positive,1
3,movie_remakes__0103,positive,1
4,movie_remakes__0104,positive,1


In [16]:
SYSTEM_PROMPT = """You are a strict narrative-structure evaluator.

Task:
Given two full story texts, rate their structural similarity from 1 to 10.

Focus on structure only:
- progression of major events
- causal/event order
- narrative stages (setup, conflict, escalation, climax, resolution)
- role correspondences between key events

Ignore:
- writing quality
- sentence-level style
- grammar
- minor wording differences

Scale:
1 = completely different structure
10 = nearly identical structural progression

Return strict JSON only:
{
  "score": <integer 1-10>,
  "brief_reason": "<1-2 sentence rationale focused on structure>"
}
"""


def _extract_json_block(text: str) -> dict[str, Any]:
    text = text.strip()
    # direct parse first
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # fallback: first {...} block
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        raise ValueError('No JSON object found in model output')
    obj = json.loads(m.group(0))
    if not isinstance(obj, dict):
        raise ValueError('Parsed JSON is not an object')
    return obj


def score_pair_with_llm(story_a: str, story_b: str) -> dict[str, Any]:
    user_prompt = (
        'Story A:\\n'
        f'{story_a}\\n\\n'
        'Story B:\\n'
        f'{story_b}\\n\\n'
        'Return only the JSON object.'
    )

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_prompt},
                ],
            )
            text = resp.choices[0].message.content or ''
            obj = _extract_json_block(text)

            score = int(obj['score'])
            if score < 1 or score > 10:
                raise ValueError(f'Invalid score: {score}')

            reason = str(obj.get('brief_reason', '')).strip()
            return {
                'score': score,
                'brief_reason': reason,
                'raw_response': text,
            }
        except Exception as e:
            last_err = e
            if attempt == MAX_RETRIES:
                break
            time.sleep(1.0 * attempt)

    raise RuntimeError(f'LLM call failed after {MAX_RETRIES} attempts: {last_err}')

In [17]:
# Resume support: if output file exists, continue from remaining pairs.
existing = {}
if RESULTS_PATH.exists():
    prev = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
    if isinstance(prev, list):
        existing = {str(x.get('pair_id')): x for x in prev if isinstance(x, dict) and x.get('pair_id')}

print('Existing scored pairs:', len(existing))

Existing scored pairs: 1


In [18]:
results = list(existing.values())
seen = set(existing.keys())

# Work only on pairs that are not already done (resume-friendly).
pending_df = movie_df[~movie_df['pair_id'].astype(str).isin(seen)].copy()

processed = 0
scored = 0
errors = 0

pbar = tqdm(
    pending_df.itertuples(index=False),
    total=len(pending_df),
    desc='LLM scoring',
    unit='pair',
)

for row in pbar:
    pair_id = str(row.pair_id)
    story_a = str(row.story_text_a)
    story_b = str(row.story_text_b)

    try:
        llm_out = score_pair_with_llm(story_a, story_b)
        rec = {
            'pair_id': pair_id,
            'dataset': row.dataset,
            'pair_type': row.pair_type,
            'label': int(row.label),
            'story_id_a': str(row.story_id_a),
            'story_id_b': str(row.story_id_b),
            'llm_structural_score_1_to_10': llm_out['score'],
            'llm_brief_reason': llm_out['brief_reason'],
            'llm_model': MODEL_NAME,
            'raw_response': llm_out['raw_response'],
        }
        results.append(rec)
        seen.add(pair_id)
        scored += 1
    except Exception as e:
        rec = {
            'pair_id': pair_id,
            'dataset': row.dataset,
            'pair_type': row.pair_type,
            'label': int(row.label),
            'story_id_a': str(row.story_id_a),
            'story_id_b': str(row.story_id_b),
            'llm_structural_score_1_to_10': None,
            'llm_brief_reason': '',
            'llm_model': MODEL_NAME,
            'error': str(e),
        }
        results.append(rec)
        seen.add(pair_id)
        errors += 1
        print(str(e))

    processed += 1

    # Save incrementally so progress is not lost.
    RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')

    pbar.set_postfix(
        processed=processed,
        scored=scored,
        errors=errors,
        skipped_initial=len(existing),
    )

    time.sleep(REQUEST_SLEEP_SECONDS)

print('Done. Total saved records:', len(results))
print('Newly processed:', processed, '| scored:', scored, '| errors:', errors)


LLM scoring:   0%|          | 0/99 [00:00<?, ?pair/s]

Done. Total saved records: 100
Newly processed: 99 | scored: 99 | errors: 0


In [20]:
# Quick summary
out_df = pd.DataFrame(results)
print('Saved file:', RESULTS_PATH)
print('Rows:', len(out_df))
print('Scored:', out_df['llm_structural_score_1_to_10'].notna().sum())
print('Errors:', out_df['llm_structural_score_1_to_10'].isna().sum())
out_df[['pair_id', 'pair_type', 'label', 'llm_structural_score_1_to_10']].head(10)

Saved file: /Users/shayan/Projects/NarrativeSimilarity/src/eval_results/movie_remakes_llm_structural_scores.json
Rows: 100
Scored: 99
Errors: 1


,pair_id,pair_type,label,llm_structural_score_1_to_10
0,movie_remakes__0100,random_negative,0,NaN
1,movie_remakes__0101,positive,1,3.0
2,movie_remakes__0102,positive,1,8.0
3,movie_remakes__0103,positive,1,10.0
4,movie_remakes__0104,positive,1,10.0
5,movie_remakes__0105,positive,1,3.0
6,movie_remakes__0106,positive,1,7.0
7,movie_remakes__0107,random_negative,0,2.0
8,movie_remakes__0108,positive,1,4.0
9,movie_remakes__0109,positive,1,9.0


In [21]:
# Average LLM structural scores by label (positive vs negative)
score_col = 'llm_structural_score_1_to_10'
tmp = out_df.copy()
tmp[score_col] = pd.to_numeric(tmp[score_col], errors='coerce')
tmp = tmp[tmp[score_col].notna()].copy()

summary = (
    tmp.groupby('label', dropna=False)[score_col]
      .agg(['count', 'mean', 'std', 'min', 'max'])
      .rename(index={0: 'negative', 1: 'positive'})
      .round(4)
)

print('Average structural score by class:')
print(summary.to_string())

if {'positive', 'negative'}.issubset(set(summary.index)):
    pos_mean = float(summary.loc['positive', 'mean'])
    neg_mean = float(summary.loc['negative', 'mean'])
    print(f"\nMean gap (positive - negative): {pos_mean - neg_mean:.4f}")


Average structural score by class:
          count    mean     std  min   max
label                                     
negative     49  2.2653  0.7576  1.0   4.0
positive     50  7.8000  2.1189  2.0  10.0

Mean gap (positive - negative): 5.5347
